# NBA Game Predictor Development

## 1. Load Data from Database

In [26]:
import pandas as pd
import sqlite3
import numpy as np # Moved here as it's used early

DB_FILE = "nba_data.db"
TABLE_NAME = "games"

# Load the dataset from SQLite
try:
    conn = sqlite3.connect(DB_FILE)
    # Ensure date column is parsed correctly when reading from SQL
    df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn, parse_dates=['date'])
    print(f"Successfully loaded {len(df)} rows from {DB_FILE}, table {TABLE_NAME}.")
except Exception as e:
    print(f"Error loading data from database: {e}")
    df = pd.DataFrame() # Create empty dataframe if loading fails
finally:
    if conn:
        conn.close()

# Sort data (important for consistent rolling calcs if done here, though ideally done before DB save)
if not df.empty:
    if 'date' in df.columns:
        # df['date'] = pd.to_datetime(df['date']) # Already parsed by read_sql_query
        df = df.sort_values(by=['date', 'team']).reset_index(drop=True) 
    else:
        print("Column 'date' not found.")

    # Display basic info and head
    print("\nDataset Info:")
    df.info()
    print("\nDataset Head:")
    print(df.head())
else:
    print("DataFrame is empty after loading attempts.")

Successfully loaded 15162 rows from nba_data.db, table games.

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15162 entries, 0 to 15161
Columns: 155 entries, Unnamed: 0 to winner
dtypes: datetime64[ns](1), float64(136), int64(8), object(10)
memory usage: 17.9+ MB

Dataset Head:
   Unnamed: 0     mp   mp.1    fg   fga    fg%    3p   3pa    3p%    ft  ...  \
0       14424  240.0  240.0  36.0  88.0  0.409   8.0  32.0  0.250  19.0  ...   
1       14425  240.0  240.0  38.0  83.0  0.458   5.0  22.0  0.227  21.0  ...   
2       11437  240.0  240.0  43.0  80.0  0.538  16.0  30.0  0.533  19.0  ...   
3       11436  240.0  240.0  47.0  97.0  0.485  15.0  41.0  0.366  13.0  ...   
4       13842  240.0  240.0  48.0  94.0  0.511   9.0  18.0  0.500  12.0  ...   

   tov%_max_opp  usg%_max_opp  ortg_max_opp  drtg_max_opp  team_opp  \
0          31.6          27.3         138.0         107.0       CLE   
1          34.7          29.9         129.0         112.0       BOS   
2         

## 2. Define Target and Select Initial Features (Pre-computation)

In [27]:
from sklearn.preprocessing import OneHotEncoder

if not df.empty:
    # Define Target Variable
    if 'winner' in df.columns:
        df['winner'] = df['winner'].astype(bool).astype(int)
        y = df['winner'].copy()
        print(f"Target variable 'y' created with shape: {y.shape}")
    else:
        print("Column 'winner' not found for target variable.")
        y = None

    # --- Store 'home' column separately for later inclusion --- 
    if 'home' in df.columns:
        home_feature = df[['home']].copy()
        print(f"'home' feature extracted with shape: {home_feature.shape}")
    else:
        print("\nWarning: 'home' column not found.")
        home_feature = pd.DataFrame(index=df.index)

    # Encode Categorical Features (Team and Opponent)
    categorical_cols = ['team', 'team_opp']
    existing_categorical_cols = [col for col in categorical_cols if col in df.columns]
    missing_categorical_cols = [col for col in categorical_cols if col not in df.columns]
    if missing_categorical_cols:
         print(f"\nWarning: Missing categorical columns: {missing_categorical_cols}")
            
    if existing_categorical_cols:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_categorical_encoded = encoder.fit_transform(df[existing_categorical_cols])
        encoded_feature_names = encoder.get_feature_names_out(existing_categorical_cols)
        X_categorical = pd.DataFrame(X_categorical_encoded, index=df.index, columns=encoded_feature_names)
        print(f"Categorical features encoded with shape: {X_categorical.shape}")
    else:
        print("\nNo categorical columns found to encode.")
        X_categorical = pd.DataFrame(index=df.index)
        encoder = None

else:
    print("DataFrame is empty. Cannot proceed with feature engineering.")
    home_feature, X_categorical, y, encoder = None, None, None, None


Target variable 'y' created with shape: (15162,)
'home' feature extracted with shape: (15162, 1)
Categorical features encoded with shape: (15162, 60)


## 3. Feature Engineering: Rolling Statistics

In [28]:
if not df.empty and 'team' in df.columns and 'date' in df.columns:
    stats_to_roll = [
        'fg%', '3p%', 'ft%', 'orb%', 'drb%', 'trb%', 'ast%', 'stl%', 'blk%', 'tov%', 
        'usg%', 'ortg', 'drtg', 
        'fg%_opp', '3p%_opp', 'ft%_opp', 'orb%_opp', 'drb%_opp', 'trb%_opp', 
        'ast%_opp', 'stl%_opp', 'blk%_opp', 'tov%_opp', 'usg%_opp', 'ortg_opp', 'drtg_opp'
    ]
    
    existing_stats_to_roll = [col for col in stats_to_roll if col in df.columns]
    missing_stats = [col for col in stats_to_roll if col not in df.columns]
    if missing_stats:
        print(f"\nWarning: Missing columns for rolling stats in df: {missing_stats}")
        
    if existing_stats_to_roll:
        N = 10 
        grouped = df.groupby('team')[existing_stats_to_roll]
        rolling_stats = grouped.rolling(window=N, closed='left').mean() 
        rolling_stats.columns = [f'{col}_roll{N}' for col in existing_stats_to_roll]
        rolling_stats = rolling_stats.reset_index(level=0, drop=True) 
        print(f"\nRolling stats calculated with shape: {rolling_stats.shape}")
        
        X = pd.concat([home_feature, X_categorical, rolling_stats], axis=1)
        X.fillna(X.median(), inplace=True)
        print(f"Features 'X' constructed with rolling stats, shape: {X.shape}")
        print(f"NaN count after imputation: {X.isna().sum().sum()}")
    else:
        print("\nNo columns found to calculate rolling stats.")
        X = pd.concat([home_feature, X_categorical], axis=1)
        X.fillna(X.median(), inplace=True)
        print(f"Features 'X' (no rolling stats) shape: {X.shape}")
else:
    print("DataFrame is empty or missing 'team'/'date' columns. Cannot calculate rolling stats.")
    if home_feature is not None and X_categorical is not None:
        X = pd.concat([home_feature, X_categorical], axis=1)
        X.fillna(X.median(), inplace=True)
        print(f"Features 'X' (no rolling stats) shape: {X.shape}")
    else:
        X = None


Rolling stats calculated with shape: (15162, 26)
Features 'X' constructed with rolling stats, shape: (15162, 87)
NaN count after imputation: 0


## 3.5 Feature Engineering: Difference Features

In [29]:
if X is not None:
    print("\nCalculating difference features...")
    diff_pairs = {
        'fg%': 'fg%_opp',
        '3p%': '3p%_opp',
        'ft%': 'ft%_opp',
        'orb%': 'orb%_opp',
        'drb%': 'drb%_opp',
        'trb%': 'trb%_opp',
        'ast%': 'ast%_opp',
        'stl%': 'stl%_opp',
        'blk%': 'blk%_opp',
        'tov%': 'tov%_opp',
        'usg%': 'usg%_opp',
        'ortg': 'drtg', 
        'ortg_opp': 'drtg_opp'
    }
    
    diff_feature_count = 0
    for team_stat, opp_stat in diff_pairs.items():
        team_roll_col = f'{team_stat}_roll{N}'
        opp_roll_col = f'{opp_stat}_roll{N}'
        diff_col_name = f'{team_stat}_diff_roll{N}'
        
        if team_roll_col in X.columns and opp_roll_col in X.columns:
            X[diff_col_name] = X[team_roll_col] - X[opp_roll_col]
            diff_feature_count += 1
        else:
            print(f"  Skipping difference for {team_stat}: Missing {team_roll_col} or {opp_roll_col}")
            
    X.fillna(X.median(), inplace=True)
    print(f"Added {diff_feature_count} difference features.")
    print(f"Features 'X' updated with difference features, shape: {X.shape}")
    print(f"NaN count after adding difference features: {X.isna().sum().sum()}")
else:
    print("\nFeature DataFrame 'X' is None, cannot calculate difference features.")


Calculating difference features...
Added 13 difference features.
Features 'X' updated with difference features, shape: (15162, 100)
NaN count after adding difference features: 0


## 4. Train Final Ensemble Model (Using All Data)

In [30]:
import xgboost as xgb 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, log_loss # Keep for potential train eval

# Use the full X and y datasets for final training
if X is not None and y is not None:
    print(f"\nTraining final model on full dataset: X={X.shape}, y={y.shape}")
    print("Setting up models for ensemble...")
    
    # --- Model 1: XGBoost --- 
    xgb_clf = xgb.XGBClassifier(objective='binary:logistic', 
                                use_label_encoder=False, 
                                eval_metric='logloss', 
                                random_state=42)
    
    # --- Model 2: Logistic Regression with Scaling --- 
    lr_pipeline = Pipeline([
        ('scaler', StandardScaler()), 
        ('logreg', LogisticRegression(max_iter=1000, 
                                     random_state=42, 
                                     class_weight='balanced'))
    ])
    
    # --- Ensemble: Voting Classifier --- 
    voting_clf = VotingClassifier(
        estimators=[('xgb', xgb_clf), ('lr', lr_pipeline)],
        voting='soft' 
    )
    
    print("Training Ensemble (VotingClassifier) on full dataset...")
    # Ensure column names are strings for XGBoost compatibility
    X.columns = X.columns.astype(str)
    
    voting_clf.fit(X, y) # Fit on the entire dataset X and y
    print("Final ensemble training complete.")
    
    # Store the final column names used for training
    final_model_columns = list(X.columns)
    print(f"Model trained with {len(final_model_columns)} features.")

    # --- Evaluation on the training set itself (optional check) --- 
    # Note: This is NOT a true test set evaluation, just shows how well it fits the training data
    # print("\nEvaluating performance on the full training set (for reference only)...")
    # y_pred_train = voting_clf.predict(X)
    # y_pred_proba_train = voting_clf.predict_proba(X)[:, 1]
    # accuracy_train = accuracy_score(y, y_pred_train)
    # ll_train = log_loss(y, y_pred_proba_train)
    # print(f"\nTraining Set Accuracy: {accuracy_train:.4f}")
    # print(f"Training Set Log Loss: {ll_train:.4f}")
    # print("\nTraining Set Classification Report:")
    # print(classification_report(y, y_pred_train))
    
else:
    print("\nTraining data (X or y) not available. Cannot train final model.")
    voting_clf = None
    final_model_columns = None


Training final model on full dataset: X=(15162, 100), y=(15162,)
Setting up models for ensemble...
Training Ensemble (VotingClassifier) on full dataset...


/opt/anaconda3/envs/algvenv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:46:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Final ensemble training complete.
Model trained with 100 features.


## 5. Save Model Artifacts

In [32]:
import joblib

MODEL_FILENAME = "nba_predictor_ensemble.joblib"
ENCODER_FILENAME = "team_encoder.joblib"
COLUMNS_FILENAME = "model_columns.joblib"

if 'voting_clf' in locals() and voting_clf is not None and \
   'encoder' in locals() and encoder is not None and \
   'final_model_columns' in locals() and final_model_columns is not None:
    try:
        print(f"\nSaving trained model to {MODEL_FILENAME}...")
        joblib.dump(voting_clf, MODEL_FILENAME)
        print("Model saved successfully.")
        
        print(f"Saving OneHotEncoder to {ENCODER_FILENAME}...")
        joblib.dump(encoder, ENCODER_FILENAME)
        print("Encoder saved successfully.")
        
        print(f"Saving model feature columns to {COLUMNS_FILENAME}...")
        joblib.dump(final_model_columns, COLUMNS_FILENAME)
        print("Columns saved successfully.")
        
    except Exception as e:
        print(f"Error saving artifacts: {e}")
else:
    print("\nModel, encoder, or columns not available. Cannot save artifacts.")


Saving trained model to nba_predictor_ensemble.joblib...
Model saved successfully.
Saving OneHotEncoder to team_encoder.joblib...
Encoder saved successfully.
Saving model feature columns to model_columns.joblib...
Columns saved successfully.
